# Batch Ground Truth Diagram Export

This notebook batch-processes multiple datasets and multiple reference conditions per dataset:

1. Apply the same preprocessing used in Urc1.
2. Select ground-truth raw points within the `Iref / Tref / OHref` windows.
3. Plot only two point sets: `all filtered voltage` and `all gt raw points`.
4. Do not export interpolated GT results.
5. Export one raw GT CSV for each reference condition.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from master_arbeit_Di.ground_truth.b_gt_distribution import (
    PreprocessConfig,
    ReferenceConditionSpec,
    export_gt_diagrams_for_many_datasets,
)

print("Project root:", PROJECT_ROOT)


Project root: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth


## Common Config

Define shared preprocessing parameters and output directories here. Most datasets can reuse the same rules. If a specific dataset needs custom filtering, adjust that dataset's config in the section below.


In [2]:
PREPROCESS_CFG = PreprocessConfig(
    i_off=0.1,
    u_off=1.3,
    resample=1,
    data_filter_i_min=0.1,
    data_filter_U_min=1.4,
    data_filter_U_max=2.3,
    data_filter_T_min=50.0,
    data_filter_T_max=65.0,
    data_filter_h_since_last_start_min=0.5,
)

PREPROCESS_OUT = r"explore_data\output"
HTML_OUT = r"output_backup\gt_html"
CSV_OUT = r"output_backup\gt_raw"
SAVE_GT_CSV = True

print("HTML output:", HTML_OUT)
print("CSV output:", CSV_OUT)


HTML output: output_backup\gt_html
CSV output: output_backup\gt_raw


## Dataset Specs

Each dataset is described by one dictionary:

- `dataset_name`: output file prefix.
- `dataset_path`: input parquet path.
- `references`: list of reference configurations and selection windows.

Each reference entry produces one independent HTML file.


In [3]:
DATASET_SPECS1 = [
    {
        "dataset_name": "G6M2",
        "dataset_path": r"..\\..\\explore_data\\G6M2.parquet",
        "references": [
            ReferenceConditionSpec(
                label="g6m2_low_load",
                iref=0.28,
                tref=57.0,
                ohref=10.0,
                i_range=(0.26, 0.30),
                t_range=(53, 60),
                oh_range=(2, 43),
            ),
            ReferenceConditionSpec(
                label="g6m2_mid_load",
                iref=1.0,
                tref=58.0,
                ohref=33.0,
                i_range=(0.98, 1.02),
                t_range=(57, 60),
                oh_range=(7, 142),
            ),
            ReferenceConditionSpec(
                label="g6m2_high_load",
                iref=1.31,
                tref=58.0,
                ohref=18.0,
                i_range=(1.29, 1.33),
                t_range=(56, 59.5),
                oh_range=(4, 78),
            ),
        ],
    }
]

print(f"Configured datasets: {len(DATASET_SPECS1)}")
for item in DATASET_SPECS1:
    print(item["dataset_name"], "->", len(item["references"]), "reference conditions")


Configured datasets: 1
G6M2 -> 3 reference conditions


## Run Batch Export

When executed, the workflow will:

1. Preprocess each dataset.
2. Extract GT raw points for each reference condition.
3. Save CSV outputs.
4. Generate a summary table with output paths and GT point counts.


In [4]:
SUMMARY_DF = export_gt_diagrams_for_many_datasets(
    dataset_specs=DATASET_SPECS1,
    preprocess_output_dir=PREPROCESS_OUT,
    output_dir=HTML_OUT,
    preprocess_config=PREPROCESS_CFG,
    save_html=True,
    save_gt_csv=SAVE_GT_CSV,
    csv_output_dir=CSV_OUT,

)

print("Generated CSV files:", len(SUMMARY_DF))


=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   explore_data\output\G6M2_20260506_165440.parquet

=== GMpreprocess Pipeline Completed Successfully ===
[preprocess_once] 1096020 -> 359353 points.
  Ground Truth Extraction: G6M2
  Reference condition:
    I  = 0.28 A/cm²   (filter: [0.26, 0.3])
    T  = 57.0 °C      (filter: [53, 60])
  Total data points:  359353
  GT points found:    30218  (8.41%)
  GT voltage range:   [1.4217, 1.6581] V
  GT time span:       2023-07-13 15:02:00 → 2025-08-05 03:08:00
  Interpolated GT coverage: 358403 / 359353 (99.7%)
  Ground Truth Extraction: G6M2
  Reference condition:
    I  = 1.0 A/cm²   (filter: [0.98, 1.02])
    T  = 58.0 °C      (filter: [57, 60])
  Tota

In [5]:
SUMMARY_DF.sort_values(["dataset_name", "iref", "tref", "ohref"]).style \
    .format({
        "iref": "{:.4f}",
        "tref": "{:.2f}",
        "ohref": "{:.2f}",
    }) \
    .background_gradient(subset=["n_gt_points"], cmap="YlOrRd")


,dataset_name,dataset_path,label,iref,tref,ohref,i_range,t_range,oh_range,n_gt_points,html_file,csv_file
0,G6M2,..\\..\\explore_data\\G6M2.parquet,g6m2_low_load,0.2800,57.00,10.00,"[0.26, 0.3]","[53, 60]","[2, 43]",30218,output_backup\gt_html\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10.html,output_backup\gt_raw\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10.csv
1,G6M2,..\\..\\explore_data\\G6M2.parquet,g6m2_mid_load,1.0000,58.00,33.00,"[0.98, 1.02]","[57, 60]","[7, 142]",597,output_backup\gt_html\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33.html,output_backup\gt_raw\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33.csv
2,G6M2,..\\..\\explore_data\\G6M2.parquet,g6m2_high_load,1.3100,58.00,18.00,"[1.29, 1.33]","[56, 59.5]","[4, 78]",504,output_backup\gt_html\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18.html,output_backup\gt_raw\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18.csv


In [6]:
DATASET_SPECS2  = [
    {
        "dataset_name": "G1M1_new",
        "dataset_path": r"..\\..\\explore_data\\G1M1_new.parquet",
        "references": [
            ReferenceConditionSpec(
                label="g1m1_low_load",
                iref=0.3,
                tref=58.0,
                ohref=11.0,
                i_range=(0.28, 0.32),
                t_range=(56, 60),
                oh_range=(1.6, 76),
            ),
            ReferenceConditionSpec(
                label="g1m1_mid_load",
                iref=1.0,
                tref=58.0,
                ohref=100.0,
                i_range=(0.98, 1.02),
                t_range=(57, 59),
                oh_range=(14, 500),
            ),
            ReferenceConditionSpec(
                label="g1m1_high_load",
                iref=1.48,
                tref=57.0,
                ohref=100.0,
                i_range=(1.46, 1.50),
                t_range=(56, 58),
                oh_range=(14, 500),
            ),
        ],
    }
]

print(f"Configured datasets: {len(DATASET_SPECS2)}")
for item in DATASET_SPECS2:
    print(item["dataset_name"], "->", len(item["references"]), "reference conditions")



Configured datasets: 1
G1M1_new -> 3 reference conditions


In [7]:
SUMMARY_DF2 = export_gt_diagrams_for_many_datasets(
    dataset_specs=DATASET_SPECS2,
    preprocess_output_dir=PREPROCESS_OUT,
    output_dir=HTML_OUT,
    preprocess_config=PREPROCESS_CFG,
    save_html=True,
    save_gt_csv=SAVE_GT_CSV,
    csv_output_dir=CSV_OUT,

)

print("Generated CSV files:", len(SUMMARY_DF2))

=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   explore_data\output\G1M1_new_20260506_165456.parquet

=== GMpreprocess Pipeline Completed Successfully ===
[preprocess_once] 2529217 -> 1069062 points.
  Ground Truth Extraction: G1M1_new
  Reference condition:
    I  = 0.3 A/cm²   (filter: [0.28, 0.32])
    T  = 58.0 °C      (filter: [56, 60])
  Total data points:  1069062
  GT points found:    388019  (36.30%)
  GT voltage range:   [1.5289, 1.6941] V
  GT time span:       2021-03-03 20:35:00 → 2025-12-09 04:06:00
  Interpolated GT coverage: 1041932 / 1069062 (97.5%)
  Ground Truth Extraction: G1M1_new
  Reference condition:
    I  = 1.0 A/cm²   (filter: [0.98, 1.02])
    T  = 58.0 °C      (fil

In [8]:
SUMMARY_DF2.sort_values(["dataset_name", "iref", "tref", "ohref"]).style \
    .format({
        "iref": "{:.4f}",
        "tref": "{:.2f}",
        "ohref": "{:.2f}",
    }) \
    .background_gradient(subset=["n_gt_points"], cmap="YlOrRd")

,dataset_name,dataset_path,label,iref,tref,ohref,i_range,t_range,oh_range,n_gt_points,html_file,csv_file
0,G1M1_new,..\\..\\explore_data\\G1M1_new.parquet,g1m1_low_load,0.3000,58.00,11.00,"[0.28, 0.32]","[56, 60]","[1.6, 76]",388019,output_backup\gt_html\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11.html,output_backup\gt_raw\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11.csv
1,G1M1_new,..\\..\\explore_data\\G1M1_new.parquet,g1m1_mid_load,1.0000,58.00,100.00,"[0.98, 1.02]","[57, 59]","[14, 500]",109407,output_backup\gt_html\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100.html,output_backup\gt_raw\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100.csv
2,G1M1_new,..\\..\\explore_data\\G1M1_new.parquet,g1m1_high_load,1.4800,57.00,100.00,"[1.46, 1.5]","[56, 58]","[14, 500]",4074,output_backup\gt_html\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100.html,output_backup\gt_raw\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100.csv


## Add More Datasets or Reference Conditions

Copy the template below and append it to `DATASET_SPECS`. Each added reference condition generates one additional HTML output.


In [9]:
# TEMPLATE = {
#     "dataset_name": "YOUR_DATASET",
#     "dataset_path": r"explore_data\YOUR_DATASET.parquet",
#     "references": [
#         ReferenceConditionSpec(
#             label="case_1",
#             iref=1.0,
#             tref=60.0,
#             ohref=72.0,
#             i_range=(0.98, 1.02),
#             t_range=(58.0, 62.0),
#             oh_range=(24.0, 160.0),
#         ),
#         ReferenceConditionSpec(
#             label="case_2",
#             iref=1.5,
#             tref=60.0,
#             ohref=120.0,
#             i_range=(1.45, 1.50),
#             t_range=(57.0, 63.0),
#             oh_range=(80.0, 200.0),
#         ),
#     ],
# }
#
# DATASET_SPECS.append(TEMPLATE)
